# Polar-vortex (SSW) diagnostics

## Purpose

Show how the initial atmosphere and land states affect the simulated 2012-13 northern polar vortex (major SSW in January 2013). Every figure uses ERA5 and ERA-Interim together: lines are the mean over the references, bands and whiskers their half-range.

## Inputs

Model: the 2012-2013 AMIP hindcasts under `INPUT_DATA_ROOT/<case>/post/atm/180x360_aave` (experiments and initial states in `configs/hind_experiment_config.py`). Monthly `ua`, `ta`, `zg` (`cmip_ts/monthly`). References: ERA5, ERA-Interim monthly (E3SM Diags time series under `OBSERVATION_ROOT`), read lazily north of 50N on the model levels.

## Method

Zonal means, then cos-lat weighted means: U over 55-65N (`u_55_65N`) and 55-75N (`u_vortex`), Z and T over 60-90N (`z_pcap`, `t_pcap`). Against each reference: bias series and RMSE profiles over all months, DJF, the SSW season (`SSW_START`-`SSW_END`), and lead-month windows from January 2012 (lead 0). Scores: RMSE change relative to CTRL by initial-state component (total vs CTRL, land effect, DA − ERA5) at `ATTRIBUTION_LEVEL`.

## Outputs

`data/hind_atm_eval/ssw/ssw_monthly_<SOURCE>_vs_<REF>.nc` (a reference's own indices: `_<REF>_vs_<REF>`), `ssw_attribution_u_<level>hPa.csv`; figures in `figure/hind_atm_eval/ssw/`:

1. Bias of U(55-65N), polar-cap Z and T at 10 and 100 hPa.
2. U bias and `RUNNING_RMSE_MONTHS`-month running RMSE.
3. RMSE attribution for `ATTRIBUTION_WINDOWS`.
4. Raw index evolution with both references and correlations.
5. Winter RMSE profiles.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
WORKFLOW_NAME = "hind_atm_eval"

EXPERIMENTS = ("CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE")
REFERENCES = ("ERA5", "ERAI")
SSW_START = (2012, 11)
SSW_END = (2013, 3)
LEAD_WINDOWS = ((0, 3), (0, 6), (10, 14))  # lead months, inclusive
ATTRIBUTION_LEVEL = 10.0                   # hPa
ATTRIBUTION_WINDOWS = ((0, 6), "ssw")      # lead months or "ssw" / "winter" / "all"
RUNNING_RMSE_MONTHS = 3
FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 300
USE_DISTRIBUTED = False  # True: local dask.distributed cluster; worker processes read files in parallel
DASK_WORKERS = 8         # threads (threaded scheduler) or worker processes (distributed); bounds memory

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.dask_helpers import compute_all, configure_dask

client = configure_dask(use_distributed=USE_DISTRIBUTED, n_workers=DASK_WORKERS, n_threads=DASK_WORKERS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, 'ssw', output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(observation_root=OBSERVATION_ROOT)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")

def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

Saved diagnostics are reused: fields are opened lazily only for the sources whose output file is missing (all of them with `FORCE_COMPUTE`), and loaded in the Derive step. An experiment is read when any of its comparison files is missing.

In [ ]:
from util.ssw_diagnostics import open_ssw_input, ssw_path

ref_paths = {r: ssw_path(data_dir, r, r) for r in REFERENCES}
exp_paths = {(e, r): ssw_path(data_dir, e, r) for e in EXPERIMENTS for r in REFERENCES}
todo = [r for r in REFERENCES if FORCE_COMPUTE or not ref_paths[r].exists()]
todo += [e for e in EXPERIMENTS if FORCE_COMPUTE or not all(exp_paths[e, r].exists() for r in REFERENCES)]
inputs = {s: open_ssw_input(s, data_root=INPUT_DATA_ROOT, **ROOTS) for s in todo}
print(f"[READ] {', '.join(inputs) or 'nothing: every index file is saved'}")

## 5. Derive

Indices of every source read, then each experiment's biases and RMSEs against each reference. The expensive reductions of every source are computed together in one Dask graph (`compute_all`), so reads and reductions run in parallel; the rest works on the loaded results.

In [ ]:
from util.ssw_diagnostics import SswAnalyzer, SswConfig

analyzer = SswAnalyzer(SswConfig(ssw_start=SSW_START, ssw_end=SSW_END, lead_windows=LEAD_WINDOWS))
indices = compute_all({s: analyzer.compute_indices(**fields, name=s, compute=False) for s, fields in inputs.items()})
ref_indices = {r: indices[r] if r in indices else xr.load_dataset(ref_paths[r]) for r in REFERENCES}
compared = {(e, r): analyzer.compare_to_reference(indices[e], ref_indices[r], r)
            for e in indices if e not in REFERENCES for r in REFERENCES}

## 6. Save

Write the newly derived files, then reload every source from disk so the steps below use the same data as a rerun that reuses the files. Check: 10-hPa U(55-65N) RMSE over the SSW season vs each reference.

In [ ]:
for r in REFERENCES:
    if r in indices:
        indices[r].assign_attrs(role="reference").to_netcdf(ref_paths[r])
        print(f"[WRITE] {ref_paths[r]}")
for key, ds in compared.items():
    ds.to_netcdf(exp_paths[key])
    print(f"[WRITE] {exp_paths[key]}")
display(xr.Dataset({r: xr.concat([xr.load_dataset(exp_paths[e, r])[f"u_55_65N_rmse_ssw_{r}"].sel(plev=1000.0)
                                  for e in EXPERIMENTS], "experiment")
                    .assign_coords(experiment=list(EXPERIMENTS)) for r in REFERENCES}).to_dataframe().round(2))

## 7. Scores

RMSE attribution of U(55-65N) at `ATTRIBUTION_LEVEL` (% of CTRL's RMSE; mean and half-range over the references).

In [ ]:
from util.ssw_plotting import SswPlotter

ssw_window = tuple(f"{y}-{m:02d}" for y, m in (SSW_START, SSW_END))
plotter = SswPlotter(data_dir, EXPERIMENTS, REFERENCES, ssw_window=ssw_window, font_scale=12)
attribution = plotter.rmse_attribution_table(p_hpa=ATTRIBUTION_LEVEL, windows=ATTRIBUTION_WINDOWS)
out = data_dir / f"ssw_attribution_u_{ATTRIBUTION_LEVEL:g}hPa.csv"
attribution.to_csv(out, index=False)
print(f"[WRITE] {out}")
display(attribution.round(1))

## 8. Plot and show

In [ ]:
save(plotter.plot_bias_timeseries(), "ssw_bias_timeseries.png")
save(plotter.plot_bias_and_running_rmse(plevs_hpa=(10.0, 100.0), window_months=RUNNING_RMSE_MONTHS),
     "ssw_u_bias_running_rmse.png")
fig, _ = plotter.plot_rmse_attribution(p_hpa=ATTRIBUTION_LEVEL, windows=ATTRIBUTION_WINDOWS, table=attribution)
save(fig, f"ssw_attribution_u_{ATTRIBUTION_LEVEL:g}hPa.png")
for p in (10.0, 100.0):
    save(plotter.plot_evolution(p), f"ssw_evolution_{p:g}hPa.png")
save(plotter.plot_rmse_profiles("winter"), "ssw_rmse_profiles_winter.png")